# Review Analysis with Routing (LLM-Powered)

**Pattern**: Route based on sentiment, with additional diagnosis for negative reviews.

**Workflow**:
```
                              ┌→ [Positive Handler] ────────────┐
START → [Sentiment Classifier] →                                 → [Reply Generator] → END
                              └→ [Negative Diagnosis] ──────────┘
```

**Flow**:
1. User provides a review
2. LLM classifies sentiment: positive or negative
3. If **positive** → simple thank you response
4. If **negative** → LLM diagnoses (extract tone, urgency, reason) → then generate reply

## Setup

In [ ]:
from typing import TypedDict, Literal
import json

from langgraph.graph import StateGraph, START, END
from langchain_aws import ChatBedrockConverse

## Initialize LLM

In [ ]:
llm = ChatBedrockConverse(
    model="us.anthropic.claude-sonnet-4-6",
    temperature=0,
)

## Define State

In [ ]:
class ReviewState(TypedDict):
    # Input
    customer_name: str
    review: str
    
    # After classification
    sentiment: str          # positive, negative
    confidence: float
    
    # After diagnosis (only for negative reviews)
    tone: str               # angry, frustrated, disappointed, sad, neutral
    urgency: str            # high, medium, low
    reason: str             # extracted reason for complaint
    key_issues: list[str]   # list of specific issues mentioned
    
    # Final output
    reply: str

## Node 1: Sentiment Classifier (LLM)

In [ ]:
def classify_sentiment(state: ReviewState) -> dict:
    """Classify review sentiment using LLM."""
    review = state["review"]
    
    prompt = f"""Analyze the sentiment of this customer review.

Review: "{review}"

Respond with ONLY a JSON object (no markdown, no explanation):
{{
    "sentiment": "positive" or "negative",
    "confidence": 0.0 to 1.0
}}"""
    
    response = llm.invoke(prompt)
    
    try:
        result = json.loads(response.content)
        return {
            "sentiment": result.get("sentiment", "negative"),
            "confidence": result.get("confidence", 0.5)
        }
    except json.JSONDecodeError:
        # Fallback if JSON parsing fails
        content = response.content.lower()
        return {
            "sentiment": "positive" if "positive" in content else "negative",
            "confidence": 0.5
        }

## Node 2a: Positive Handler (Simple Path)

In [ ]:
def positive_handler(state: ReviewState) -> dict:
    """Handle positive reviews - set default values."""
    return {
        "tone": "happy",
        "urgency": "low",
        "reason": "Customer is satisfied",
        "key_issues": []
    }

## Node 2b: Negative Diagnosis (LLM - Extract Tone, Urgency, Reason)

In [ ]:
def diagnose_negative(state: ReviewState) -> dict:
    """Diagnose negative review using LLM - extract tone, urgency, reason, and issues."""
    review = state["review"]
    
    prompt = f"""Analyze this negative customer review and extract key information.

Review: "{review}"

Respond with ONLY a JSON object (no markdown, no explanation):
{{
    "tone": "angry" | "frustrated" | "disappointed" | "sad" | "neutral",
    "urgency": "high" | "medium" | "low",
    "reason": "brief summary of the main complaint",
    "key_issues": ["issue1", "issue2", ...]
}}

Guidelines:
- tone: angry (threats, caps, exclamations), frustrated (repeated attempts failed), disappointed (expectations not met), sad (emotional), neutral (factual complaint)
- urgency: high (legal threats, immediate action demanded), medium (waiting for response), low (general feedback)
- key_issues: specific problems like "product quality", "delivery delay", "customer service", "pricing", "false advertising"""
    
    response = llm.invoke(prompt)
    
    try:
        result = json.loads(response.content)
        return {
            "tone": result.get("tone", "neutral"),
            "urgency": result.get("urgency", "low"),
            "reason": result.get("reason", "Customer complaint"),
            "key_issues": result.get("key_issues", ["General Dissatisfaction"])
        }
    except json.JSONDecodeError:
        # Fallback if JSON parsing fails
        return {
            "tone": "neutral",
            "urgency": "medium",
            "reason": "Customer expressed dissatisfaction",
            "key_issues": ["General Dissatisfaction"]
        }

## Node 3: Reply Generator (LLM)

In [ ]:
def generate_reply(state: ReviewState) -> dict:
    """Generate appropriate reply using LLM based on sentiment and diagnosis."""
    
    if state["sentiment"] == "positive":
        prompt = f"""Write a brief, warm thank-you response to this positive customer review.

Customer Name: {state['customer_name']}
Review: "{state['review']}"

Keep it professional, 3-4 sentences max. Sign off as "Customer Success Team"."""
    else:
        prompt = f"""Write a professional response to this negative customer review.

Customer Name: {state['customer_name']}
Review: "{state['review']}"

Analysis:
- Tone: {state['tone']}
- Urgency: {state['urgency']}
- Main Issues: {', '.join(state['key_issues'])}
- Reason: {state['reason']}

Guidelines:
- Match empathy level to the tone (more apologetic for angry customers)
- If urgency is high, mention immediate escalation
- Address the specific issues mentioned
- Offer concrete next steps
- Keep it professional, 5-7 sentences
- Sign off as "Customer Resolution Team"""
    
    response = llm.invoke(prompt)
    
    # Build final output
    lines = []
    lines.append("=" * 60)
    lines.append("              REVIEW ANALYSIS & RESPONSE")
    lines.append("=" * 60)
    
    lines.append(f"\n[CUSTOMER] {state['customer_name']}")
    lines.append(f"\n[REVIEW] \"{state['review']}\"")
    
    lines.append("\n" + "-" * 60)
    lines.append("ANALYSIS")
    lines.append("-" * 60)
    lines.append(f"  Sentiment: {state['sentiment'].upper()}")
    lines.append(f"  Confidence: {state['confidence']*100:.0f}%")
    
    if state["sentiment"] == "negative":
        lines.append(f"  Tone: {state['tone'].upper()}")
        lines.append(f"  Urgency: {state['urgency'].upper()}")
        lines.append(f"  Reason: {state['reason']}")
        lines.append(f"  Issues: {', '.join(state['key_issues'])}")
    
    lines.append("\n" + "-" * 60)
    lines.append("GENERATED REPLY")
    lines.append("-" * 60)
    lines.append(f"\n{response.content}")
    lines.append("\n" + "=" * 60)
    
    return {"reply": "\n".join(lines)}

## Routing Function

In [ ]:
def route_by_sentiment(state: ReviewState) -> Literal["positive", "negative"]:
    """Route to appropriate handler based on sentiment."""
    if state["sentiment"] == "positive":
        return "positive"
    else:
        return "negative"

## Build the Graph

In [ ]:
# Create graph
graph = StateGraph(ReviewState)

# Add nodes
graph.add_node("classifier", classify_sentiment)
graph.add_node("positive_handler", positive_handler)
graph.add_node("diagnose_negative", diagnose_negative)
graph.add_node("generate_reply", generate_reply)

# Add edges
graph.add_edge(START, "classifier")

# CONDITIONAL EDGE: Route based on sentiment
graph.add_conditional_edges(
    "classifier",
    route_by_sentiment,
    {
        "positive": "positive_handler",
        "negative": "diagnose_negative"
    }
)

# Both paths lead to reply generator
graph.add_edge("positive_handler", "generate_reply")
graph.add_edge("diagnose_negative", "generate_reply")

graph.add_edge("generate_reply", END)

# Compile
app = graph.compile()

## Visualize the Graph

In [ ]:
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

## Test Case 1: Positive Review

In [ ]:
result = app.invoke({
    "customer_name": "Rahul Sharma",
    "review": "Absolutely love this product! Amazing quality and fast delivery. Best purchase ever!"
})

print(result["reply"])

## Test Case 2: Negative Review - Angry Customer (High Urgency)

In [ ]:
result = app.invoke({
    "customer_name": "Priya Patel",
    "review": "This is unacceptable! The product arrived broken and customer service hung up on me. I want a refund immediately or I'm calling my lawyer!"
})

print(result["reply"])

## Test Case 3: Negative Review - Frustrated Customer

In [ ]:
result = app.invoke({
    "customer_name": "Amit Kumar",
    "review": "I'm so frustrated! I've tried everything but the app keeps crashing. Been waiting for support for 3 days now."
})

print(result["reply"])

## Test Case 4: Negative Review - Disappointed Customer

In [ ]:
result = app.invoke({
    "customer_name": "Sneha Reddy",
    "review": "Really disappointed with this purchase. Expected much better quality based on the description. Not what I thought it would be."
})

print(result["reply"])

## Test Case 5: Negative Review - Delivery Issue

In [ ]:
result = app.invoke({
    "customer_name": "Vikram Singh",
    "review": "Terrible service! My order was delayed by 2 weeks and when it finally arrived, it was sent to the wrong address. Very poor experience."
})

print(result["reply"])

## Test Case 6: Mixed Review

In [ ]:
result = app.invoke({
    "customer_name": "Ananya Gupta",
    "review": "The product looks good, but it stopped working after just 2 days. Waste of money."
})

print(result["reply"])

## Stream Execution (See the Path)

In [ ]:
print("=" * 60)
print("STREAMING EXECUTION")
print("=" * 60)

for step in app.stream({
    "customer_name": "Test User", 
    "review": "Horrible product! Overpriced and doesn't work. I want my money back now!"
}):
    for node_name, output in step.items():
        print(f"\n[DONE] Node: {node_name}")
        
        if node_name == "classifier":
            print(f"   Sentiment: {output['sentiment']} ({output['confidence']*100:.0f}%)")
            print(f"   -> Routing to: {'positive_handler' if output['sentiment'] == 'positive' else 'diagnose_negative'}")
        
        elif node_name == "diagnose_negative":
            print(f"   Tone: {output['tone']}")
            print(f"   Urgency: {output['urgency']}")
            print(f"   Issues: {output['key_issues']}")
        
        elif node_name == "positive_handler":
            print("   -> Simple positive flow (no diagnosis needed)")

## Interactive: Try Your Own Review

In [ ]:
# Change these!
my_name = "Your Name"
my_review = "Write your review here..."

result = app.invoke({
    "customer_name": my_name,
    "review": my_review
})

print(result["reply"])

## Key Takeaways

1. **LLM-powered classification** is more accurate than keyword matching
2. **LLM diagnosis** extracts nuanced information (tone, urgency, specific issues)
3. **LLM reply generation** creates natural, contextual responses
4. **Conditional routing** still works the same - route function reads LLM output
5. **JSON output parsing** with fallback ensures robustness

### LLM Integration Pattern:
```python
def node_with_llm(state):
    prompt = f"...{state['field']}..."
    response = llm.invoke(prompt)
    result = json.loads(response.content)  # Parse structured output
    return {"field": result["key"]}
```